In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-17 12:22:36--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.1’

book-crossings.zip. 100%[===================>]  24.88M   160MB/s    in 0.2s    

2026-05-17 12:22:36 (160 MB/s) - ‘book-crossings.zip.1’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code
# Load the data
books = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip',
                    usecols=['ISBN', 'Book-Title', 'Book-Author'])
ratings = pd.read_csv('BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

# Rename columns
books.columns = ['isbn', 'title', 'author']
ratings.columns = ['user', 'isbn', 'rating']

# Filter users with less than 200 ratings
user_counts = ratings['user'].value_counts()
ratings = ratings[ratings['user'].isin(user_counts[user_counts >= 200].index)]

# Filter books with less than 100 ratings
book_counts = ratings['isbn'].value_counts()
ratings = ratings[ratings['isbn'].isin(book_counts[book_counts >= 100].index)]

# Merge ratings with book titles
df = ratings.merge(books, on='isbn')

# Remove duplicates
df = df.drop_duplicates(['user', 'title'])

# Create pivot table
book_pivot = df.pivot(index='title', columns='user', values='rating').fillna(0)

# Convert to sparse matrix
from scipy.sparse import csr_matrix
book_sparse = csr_matrix(book_pivot.values)

# Train KNN model
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_sparse)

# Recommendation function
def get_recommends(book=""):
    book_idx = book_pivot.index.get_loc(book)

    distances, indices = model.kneighbors(
        book_pivot.iloc[book_idx, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []
    for i in range(1, len(distances.flatten())):
        recommended_books.append([
            book_pivot.index[indices.flatten()[i]],
            distances.flatten()[i]
        ])

    # Reverse so closest is last
    recommended_books = recommended_books[::-1]

    return [book, recommended_books]

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):


  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()